# Week 21, Day 3: Git: commits and history

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w021d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
versions = []
essay = "Intro"
versions.append(("first draft", essay))
essay += "\nMain point"
versions.append(("add main point", essay))
essay += "\nConclusion"
versions.append(("add conclusion", essay))

for message, text in versions:
    print(message, "->", text.count("\n") + 1, "lines")
print(versions[0][1])

*Interactive figure on the website: A practice terminal with Git. Work through the tasks: each turns green when it's done. Use git status often: it always says what to do next..*

> **Key idea.** Git records snapshots called commits. The cycle is: change files, `git status`, `git add`, `git commit -m "message"`. `git log` shows the history and `git diff` shows changes. Make each commit one logical change with a clear message, and keep secrets and generated files out with `.gitignore`.

### Exercise: Snapshots

A tiny version store keeps a list of commits, each a dictionary `{"message": ..., "files": {...}}` holding a **copy** of the files at that moment. Write `commit(history, files, message)` that appends a new commit with a copy of `files` and returns its number (0 for the first), and `checkout(history, number)` that returns a **copy** of the files from that commit. Changing the working files later must not change any commit.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    history = []\n    files = {\"a.txt\": \"one\"}\n    assert commit(history, files, \"first\") == 0\n    files[\"a.txt\"] = \"two\"\n    files[\"b.txt\"] = \"new\"\n    assert commit(history, files, \"second\") == 1\n    assert checkout(history, 0) == {\"a.txt\": \"one\"}\n    old = checkout(history, 0)\n    old[\"a.txt\"] = \"changed\"\n    assert checkout(history, 0) == {\"a.txt\": \"one\"} and history[1][\"message\"] == \"second\"")

### Exercise: git status

Write `status(head, index, working)` where each argument is a dictionary from file name to contents (the last commit, the staging area, and the working folder). Return a dictionary with three sorted lists: `"staged"` (files whose content differs between `head` and `index`, including new and deleted), `"unstaged"` (files in `index` whose working content differs, or that are missing from working) and `"untracked"` (files in working but not in index).

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    head = {\"a.py\": \"1\", \"b.py\": \"1\", \"c.py\": \"1\"}\n    index = {\"a.py\": \"2\", \"b.py\": \"1\", \"c.py\": \"1\", \"new.py\": \"x\"}\n    working = {\"a.py\": \"2\", \"b.py\": \"9\", \"new.py\": \"x\", \"notes.txt\": \"hi\"}\n    assert status(head, index, working) == {\"staged\": [\"a.py\", \"new.py\"], \"unstaged\": [\"b.py\", \"c.py\"], \"untracked\": [\"notes.txt\"]}\n    same = {\"a\": \"1\"}\n    assert status(same, same, same) == {\"staged\": [], \"unstaged\": [], \"untracked\": []}\n    assert status({\"a\": \"1\"}, {}, {\"a\": \"1\"})[\"staged\"] == [\"a\"]")

### Exercise: Commit IDs

Git names each commit by a hash of its contents, so the same contents always give the same ID and any change gives a different one. Write `commit_id(files, message, parent)` returning the first 7 hexadecimal characters of the SHA-1 hash (`hashlib.sha1(text.encode()).hexdigest()`) of a text made from the parent's ID (or `"none"`), the message, and each file's name and contents, **in name order** so that the order of the dictionary doesn't matter. Use exactly this text: the lines `parent <id>`, `message <msg>`, then `file <name> <contents>` for each file, joined with newlines.

In [ ]:
import hashlib

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    a = commit_id({\"a.txt\": \"hi\", \"b.txt\": \"yo\"}, \"first\", None)\n    b = commit_id({\"b.txt\": \"yo\", \"a.txt\": \"hi\"}, \"first\", None)\n    assert a == b and len(a) == 7 and all(ch in \"0123456789abcdef\" for ch in a)\n    assert commit_id({\"a.txt\": \"hi!\", \"b.txt\": \"yo\"}, \"first\", None) != a\n    assert commit_id({\"a.txt\": \"hi\", \"b.txt\": \"yo\"}, \"first\", a) != a\n    import hashlib\n    assert a == hashlib.sha1(\"parent none\\nmessage first\\nfile a.txt hi\\nfile b.txt yo\".encode()).hexdigest()[:7]")

### Exercise: git diff

Write `diff(old, new)` comparing two texts line by line and returning the changed lines as a list: `"- line"` for a removed line and `"+ line"` for an added one, in order, leaving out unchanged lines. Use `difflib.ndiff(old.splitlines(), new.splitlines())`, which produces lines starting with `"- "`, `"+ "`, `"  "` (unchanged) or `"? "` (hints, to skip).

In [ ]:
import difflib

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    old = \"Intro\\nMain point\\nEnd\\n\"\n    new = \"Intro\\nMain argument\\nEnd\\nThanks\\n\"\n    assert diff(old, new) == [\"- Main point\", \"+ Main argument\", \"+ Thanks\"]\n    assert diff(\"a\\nb\\n\", \"a\\nb\\n\") == [] and diff(\"\", \"x\\n\") == [\"+ x\"]")

### Exercise: Good messages

Write `good_message(msg)` that returns `True` for a commit message whose first line is between 8 and 50 characters, starts with a capital letter, doesn't end with a full stop, and isn't one of the lazy messages `"update"`, `"changes"`, `"fix"`, `"stuff"`, `"wip"` (ignoring case).

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert good_message(\"Add a word count to the report\")\n    assert good_message(\"Fix crash on empty file\\n\\nThe parser assumed one line at least.\")\n    assert not good_message(\"stuff\") and not good_message(\"Fixed it.\") and not good_message(\"add search\")\n    assert not good_message(\"Update\") and not good_message(\"X\" * 51) and not good_message(\"Fix bug\")")

### Bug hunt: Commits that change by themselves

This version store saves the **same** dictionary in every commit, so when the files change, every old commit changes too. It should print `one` and then `two`. Fix `commit`.

In [ ]:
history = []

def commit(files, message):
    history.append({"message": message, "files": files})

files = {"a.txt": "one"}
commit(files, "first")
files["a.txt"] = "two"
commit(files, "second")
print(history[0]["files"]["a.txt"])
print(history[1]["files"]["a.txt"])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"snapshots\"\n    expect_output(\"one\\ntwo\")")

### Bug hunt: What not to commit

This `.gitignore` text is meant to keep out Python's cache folders, the virtual environment folder `.venv`, log files and the secrets file `.env`, but two lines are wrong, so `ignored(".env")` and `ignored("app.log")` are `False`. Fix the two lines in `GITIGNORE`.

In [ ]:
import fnmatch

GITIGNORE = """__pycache__/
.venv/
log
env
"""

def ignored(name):
    for pattern in GITIGNORE.split():
        if pattern.endswith("/"):
            if name.startswith(pattern):
                return True
        elif fnmatch.fnmatch(name, pattern):
            return True
    return False

print([ignored(n) for n in [".env", "app.log", "__pycache__/x.pyc", ".venv/bin/python", "main.py"]])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fixed\"\n    expect_output(\"[True, True, True, True, False]\")")

### Puzzle: The everyday cycle

Arrange the Git commands in the order you'd use them to start tracking a folder and make two commits. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Answer from what you know about Git.

In [ ]:
a = ___   # the command that shows what has changed and what's staged
b = ___   # the area between your files and the history (two words, lower case)
c = ___   # True or False: git add saves a commit
d = ___   # the file that lists things Git should ignore (its name)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == \"git status\" and b == \"staging area\" and c is False and d == \".gitignore\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a tiny Git

Write a small version-control store as a set of functions over a dictionary `repo` made by `init()`: `{"commits": {}, "head": None, "index": {}}`.

- `add(repo, working, names)`: copy the named files from the `working` dictionary into the index (a name missing from `working` is removed from the index).
- `commit(repo, message)`: if the index equals the head commit's files, return `None` ("nothing to commit"); otherwise make a commit `{"files": copy of index, "message": ..., "parent": head id}`, with an ID from `commit_id` (as in the hashing exercise, given in the starter), store it under that ID, move `head` to it, and return the ID.
- `log(repo)`: a list of `"id message"` strings from the head back to the first commit.
- `show(repo, cid, name)`: the contents of file `name` as it was in commit `cid`, or `None`.

In [ ]:
import hashlib

def commit_id(files, message, parent):
    lines = [f"parent {parent or 'none'}", f"message {message}"]
    for name in sorted(files):
        lines.append(f"file {name} {files[name]}")
    return hashlib.sha1("\n".join(lines).encode()).hexdigest()[:7]

def init():
    return {"commits": {}, "head": None, "index": {}}

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_flow():\n    \"a session\"\n    repo = init()\n    work = {\"essay.txt\": \"Intro\\n\", \"notes.txt\": \"ideas\"}\n    add(repo, work, [\"essay.txt\"])\n    c1 = commit(repo, \"Start the essay\")\n    assert c1 and repo[\"head\"] == c1 and commit(repo, \"again\") is None\n    work[\"essay.txt\"] += \"Main point\\n\"\n    add(repo, work, [\"essay.txt\", \"notes.txt\"])\n    c2 = commit(repo, \"Add main point and notes\")\n    assert log(repo) == [f\"{c2} Add main point and notes\", f\"{c1} Start the essay\"]\n    assert show(repo, c1, \"essay.txt\") == \"Intro\\n\" and show(repo, c1, \"notes.txt\") is None\n    assert show(repo, c2, \"notes.txt\") == \"ideas\"\n    del work[\"notes.txt\"]\n    add(repo, work, [\"notes.txt\"])\n    c3 = commit(repo, \"Remove notes\")\n    assert show(repo, c3, \"notes.txt\") is None and len(log(repo)) == 3\n    work[\"essay.txt\"] = \"changed\"\n    assert show(repo, c2, \"essay.txt\") == \"Intro\\nMain point\\n\"")

## Warm-up

That's the idea of Git in eight lines: every time you're happy with your work, save a **snapshot** with a **message**, and keep them all. Later you can see every version, and get any of them back.

## Snapshots

- A **repository** (or "repo") is a project folder that Git is tracking. Git keeps its records in a hidden folder inside it called `.git`.
- A **commit** is a snapshot of the tracked files at one moment, with a message, the author, the time, and a link to the commit before it (its **parent**). Commits form a history going back to the first.
- Each commit has an ID, a long hexadecimal **hash** like `3f9a2c1e...` worked out from its contents. People usually use the first 7 characters.

Between your files and the history sits the **staging area** (also called the index): the set of changes you've chosen to go into the **next** commit. That lets you commit some changes and not others.

```
working folder  --git add-->  staging area  --git commit-->  history
  (your files)                (next snapshot)                (all snapshots)
```

## The everyday cycle

1. `git init` once, to start tracking a folder.
2. Make changes to your files.
3. `git status` to see what's changed.
4. `git add file` (or `git add .` for everything) to stage changes.
5. `git commit -m "message"` to save the snapshot.
6. Repeat from 2.

Try it in this practice terminal, which has a small Git built in (it does the common commands; the real Git does far more). The tasks below it walk you through the cycle.

`git status` is the command to type whenever you're unsure: it says which branch you're on, what's staged, what's changed but not staged, and which files Git isn't tracking yet. It even suggests what to do next.

## Looking back and undoing

| Command | Shows or does |
|---|---|
| `git log` | the history, newest first |
| `git log --oneline` | one line per commit |
| `git diff` | changes not yet staged (working folder vs staging area) |
| `git diff --staged` | changes staged for the next commit (vs the last commit) |
| `git show <id>` | one commit and its changes |
| `git restore file` | throw away unstaged changes to a file (careful: they're gone) |
| `git restore --staged file` | unstage a file, keeping your changes |

`git diff` shows changes **line by line**: lines starting with `-` were removed and lines starting with `+` were added. A changed line shows as one of each.

Going back to an old version of a file: `git restore --source <id> file` brings back that file as it was in commit `<id>`. Undoing a whole commit that's already shared: `git revert <id>` makes a **new** commit that reverses it, so the history stays honest. (You'll meet more ways of undoing as you need them; the important thing is that once work is committed, it's very hard to lose.)

## Good commits

A commit should be **one logical change**: "Add search box", "Fix crash on empty file". Small, focused commits make the history easy to read, and make it easy to find (and undo) the change that broke something.

A good commit message:

- has a short first line (under about 50 characters) saying **what** the commit does, written as a command: "Add", "Fix", "Remove", "Rename", not "Added stuff";
- if needed, after a blank line, says **why**.

| Poor | Better |
|---|---|
| `stuff` | `Add a word count to the report` |
| `fixed it` | `Fix crash when the marks file is empty` |
| `changes` | `Rename total() to total_price()` |
| `WIP final final` | `Split parsing into its own module` |

Some files should **never** be committed: passwords and API keys, large data files, and files your tools generate (`__pycache__/`, `.venv/`). List them in a file called `.gitignore` in the project folder, one pattern per line, and Git will leave them alone:

```
__pycache__/
.venv/
*.log
.env
```

## On your own computer

1. **Install Git.** macOS: type `git --version` in Terminal and accept the offer to install the developer tools. Windows: download it from git-scm.com (this also gives you Git Bash). Linux: it's usually there already, or use your package manager.
2. **Tell Git who you are** (once per computer; this goes in every commit):
   ```
   git config --global user.name "Your Name"
   git config --global user.email "you@example.com"
   git config --global init.defaultBranch main
   ```
3. **Make your first repository:**
   ```
   cd ~/atlas
   git init
   git status
   git add .
   git commit -m "Start my Python Atlas work"
   git log --oneline
   ```
4. From now on, commit your course work at the end of every day.

## Practice

These exercises build the ideas behind Git in Python: snapshots, staging, history, hashes and diffs.

## Quiz

## Daily challenge

## Before you go

Make a habit of it: at the end of each day's work, `git status`, `git add .`, and `git commit -m` with a message saying what you did. Look at `git log --oneline` at the end of the week to see your progress. Tomorrow: branches, which let you try an idea without disturbing work that's already going well.